# 基于强化学习与生成模型的室外岛屿地图自动生成 - 完整版

本Notebook包含完整的实验流程：
1. PCG基座模块
2. 结构评估模块
3. β-VAE表征学习
4. SAC强化学习训练
5. 实验评估与可视化

## 环境准备

In [ ]:
# 导入必要的库
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import torch
from torch.utils.data import DataLoader
import sys
import os
from tqdm import tqdm
import time

# 添加当前目录到路径
sys.path.append(os.getcwd())

# 导入自定义模块
from pcg_generator import PCGIslandGenerator
from structure_evaluator import StructureEvaluator
from vae_model import BetaVAE, HeightmapDataset, train_vae
from rl_environment import IslandGenerationEnv
from sac_agent import SACAgent, ReplayBuffer

# 设置随机种子
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# 设备配置
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")
print("✅ 所有模块导入成功！")

## 第一部分：数据集生成

In [ ]:
# 生成训练数据集
print("生成训练数据集...")
generator = PCGIslandGenerator(map_size=64)

n_samples = 500  # Demo用500，完整版可用5000+
heightmaps = []

for i in tqdm(range(n_samples)):
    params = {
        'f': np.random.uniform(5, 20),
        'A': np.random.uniform(0.5, 1.5),
        'N_octaves': np.random.randint(3, 6),
        'persistence': np.random.uniform(0.3, 0.7),
        'lacunarity': np.random.uniform(1.5, 2.5),
        'seed': i,
        'warp_strength': np.random.uniform(0.2, 0.8),
        'warp_frequency': np.random.uniform(1, 5),
        'falloff_radius': np.random.uniform(20, 40),
        'falloff_exponent': np.random.uniform(1.5, 3)
    }
    heightmap = generator.generate_heightmap(params)
    heightmaps.append(heightmap)

heightmaps = np.array(heightmaps)
print(f"✅ 生成了 {n_samples} 个高度图，形状: {heightmaps.shape}")

# 可视化部分样本
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.ravel()

for i in range(10):
    axes[i].imshow(heightmaps[i], cmap='terrain')
    axes[i].set_title(f'Sample {i+1}')
    axes[i].axis('off')

plt.tight_layout()
plt.savefig('dataset_samples.png', dpi=150, bbox_inches='tight')
plt.show()

## 第二部分：β-VAE训练

In [ ]:
# 创建数据集和数据加载器
dataset = HeightmapDataset(heightmaps)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# 创建β-VAE模型
latent_dim = 32
beta = 4.0
vae = BetaVAE(map_size=64, latent_dim=latent_dim, beta=beta).to(device)

print(f"VAE模型参数数量: {sum(p.numel() for p in vae.parameters()):,}")
print(f"隐空间维度: {latent_dim}")
print(f"β值: {beta}")

In [ ]:
# 训练VAE
print("开始训练β-VAE...")
vae_epochs = 50  # Demo用50，完整版可用100+
vae_lr = 1e-3

start_time = time.time()
vae_loss_history = train_vae(vae, dataloader, epochs=vae_epochs, 
                             learning_rate=vae_lr, device=device)
training_time = time.time() - start_time

print(f"\n训练完成！耗时: {training_time:.2f}秒")

# 绘制训练曲线
epochs_list = [h['epoch'] for h in vae_loss_history]
total_losses = [h['total_loss'] for h in vae_loss_history]
recon_losses = [h['recon_loss'] for h in vae_loss_history]
kl_losses = [h['kl_loss'] for h in vae_loss_history]

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_list, total_losses, label='Total Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('VAE Training Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epochs_list, recon_losses, label='Reconstruction', linewidth=2)
plt.plot(epochs_list, kl_losses, label='KL Divergence', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss Components')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('vae_training_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ VAE训练完成！")

In [ ]:
# 测试VAE重建效果
vae.eval()
test_maps = torch.FloatTensor(heightmaps[:8]).unsqueeze(1).to(device)  # (8, 1, 64, 64)

with torch.no_grad():
    reconstructed, mu, logvar = vae(test_maps)
    reconstructed = reconstructed.cpu().numpy()

# 可视化原始和重建的地图
fig, axes = plt.subplots(2, 8, figsize=(16, 4))

for i in range(8):
    # 原始
    axes[0, i].imshow(heightmaps[i], cmap='terrain')
    axes[0, i].set_title(f'Original {i+1}')
    axes[0, i].axis('off')
    
    # 重建
    axes[1, i].imshow(reconstructed[i, 0], cmap='terrain')
    axes[1, i].set_title(f'Reconstructed {i+1}')
    axes[1, i].axis('off')

plt.suptitle('VAE Reconstruction Quality', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('vae_reconstruction.png', dpi=150, bbox_inches='tight')
plt.show()

# 计算重建误差
mse = np.mean((heightmaps[:8] - reconstructed[:, 0])**2)
print(f"平均重建误差 (MSE): {mse:.6f}")
print("✅ VAE重建测试完成！")

## 第三部分：SAC强化学习训练

In [ ]:
# 创建环境
env = IslandGenerationEnv(map_size=64, max_steps=30)

# 创建SAC智能体
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]

agent = SACAgent(
    state_dim=state_dim,
    action_dim=action_dim,
    hidden_dim=256,
    learning_rate=3e-4,
    gamma=0.99,
    tau=0.005,
    alpha=0.2,
    action_range=0.1
).to(device)

print(f"状态维度: {state_dim}")
print(f"动作维度: {action_dim}")
print(f"SAC智能体参数数量: {sum(p.numel() for p in agent.q_network.parameters()) + sum(p.numel() for p in agent.policy.parameters()):,}")

In [ ]:
# 训练SAC
print("开始SAC训练...")

# 超参数
n_episodes = 100  # Demo用100，完整版可用500+
batch_size = 64
buffer_capacity = 10000
min_buffer_size = 1000

# 初始化回放缓冲区
replay_buffer = ReplayBuffer(capacity=buffer_capacity)

# 记录训练数据
episode_rewards = []
episode_lengths = []
losses_history = []

start_time = time.time()

for episode in tqdm(range(n_episodes)):
    state, _ = env.reset(seed=episode)
    episode_reward = 0
    episode_step = 0
    
    for step in range(env.max_steps):
        # 选择动作
        action = agent.select_action(state)
        
        # 执行动作
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        
        # 存储经验
        replay_buffer.push(state, action, reward, next_state, done)
        
        # 更新网络
        if len(replay_buffer) >= min_buffer_size:
            losses = agent.update(replay_buffer, batch_size=batch_size)
            losses_history.append(losses)
        
        episode_reward += reward
        episode_step += 1
        state = next_state
        
        if done:
            break
    
    episode_rewards.append(episode_reward)
    episode_lengths.append(episode_step)
    
    # 打印进度
    if (episode + 1) % 10 == 0:
        avg_reward = np.mean(episode_rewards[-10:])
        avg_length = np.mean(episode_lengths[-10:])
        print(f"Episode {episode+1}/{n_episodes}, "
              f"Avg Reward: {avg_reward:.4f}, "
              f"Avg Length: {avg_length:.1f}")

training_time = time.time() - start_time
print(f"\n训练完成！总耗时: {training_time:.2f}秒")
print(f"平均每episode耗时: {training_time/n_episodes:.2f}秒")

In [ ]:
# 绘制训练曲线
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Episode奖励
axes[0, 0].plot(episode_rewards, alpha=0.5, label='Raw')
if len(episode_rewards) >= 10:
    moving_avg = np.convolve(episode_rewards, np.ones(10)/10, mode='valid')
    axes[0, 0].plot(range(9, len(episode_rewards)), moving_avg, 
                    label='10-Episode MA', linewidth=2, color='red')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Total Reward')
axes[0, 0].set_title('Episode Rewards')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Episode长度
axes[0, 1].plot(episode_lengths, alpha=0.5)
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('Steps')
axes[0, 1].set_title('Episode Lengths')
axes[0, 1].grid(True, alpha=0.3)

# 3. Q损失和策略损失
if losses_history:
    q_losses = [l['q_loss'] for l in losses_history]
    policy_losses = [l['policy_loss'] for l in losses_history]
    
    # 平滑处理
    window = 50
    if len(q_losses) >= window:
        q_losses_smooth = np.convolve(q_losses, np.ones(window)/window, mode='valid')
        policy_losses_smooth = np.convolve(policy_losses, np.ones(window)/window, mode='valid')
        
        axes[1, 0].plot(q_losses_smooth, label='Q Loss', linewidth=2)
        axes[1, 0].set_xlabel('Update Step')
        axes[1, 0].set_ylabel('Loss')
        axes[1, 0].set_title('Q Network Loss')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        axes[1, 1].plot(policy_losses_smooth, label='Policy Loss', linewidth=2, color='orange')
        axes[1, 1].set_xlabel('Update Step')
        axes[1, 1].set_ylabel('Loss')
        axes[1, 1].set_title('Policy Network Loss')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('sac_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ 训练曲线已保存！")

## 第四部分：模型评估

In [ ]:
# 使用训练好的策略生成多个岛屿
print("生成测试岛屿...")
n_test_islands = 20
test_islands = []
test_metrics_list = []

for i in range(n_test_islands):
    env = IslandGenerationEnv(map_size=64, max_steps=30)
    state, _ = env.reset(seed=i*10)
    
    for step in range(env.max_steps):
        action = agent.select_action(state, evaluate=True)  # 确定性策略
        state, reward, terminated, truncated, info = env.step(action)
        
        if terminated or truncated:
            break
    
    test_islands.append(info['heightmap'])
    test_metrics_list.append(info['metrics'])

print(f"✅ 生成了 {n_test_islands} 个测试岛屿")

In [ ]:
# 统计分析
print("\n📊 测试岛屿质量统计:")
print("=" * 60)

metric_keys = list(test_metrics_list[0].keys())
for key in metric_keys:
    values = [m[key] for m in test_metrics_list]
    mean_val = np.mean(values)
    std_val = np.std(values)
    min_val = np.min(values)
    max_val = np.max(values)
    print(f"{key:25s}: {mean_val:.4f} ± {std_val:.4f}  [{min_val:.4f}, {max_val:.4f}]")

# 可视化指标分布
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for idx, key in enumerate(metric_keys[:6]):
    values = [m[key] for m in test_metrics_list]
    axes[idx].hist(values, bins=10, edgecolor='black', alpha=0.7)
    axes[idx].axvline(np.mean(values), color='red', linestyle='--', 
                      linewidth=2, label=f'Mean: {np.mean(values):.3f}')
    axes[idx].set_xlabel(key)
    axes[idx].set_ylabel('Frequency')
    axes[idx].set_title(f'Distribution of {key}')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('metrics_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 展示生成的岛屿
fig, axes = plt.subplots(4, 5, figsize=(15, 12))
axes = axes.ravel()

for i in range(min(20, n_test_islands)):
    im = axes[i].imshow(test_islands[i], cmap='terrain')
    conn = test_metrics_list[i]['connectivity']
    nav = test_metrics_list[i]['navigable_ratio']
    axes[i].set_title(f'#{i+1}\nC={conn:.2f}, N={nav:.2f}', fontsize=8)
    axes[i].axis('off')

plt.suptitle('Generated Islands by SAC Agent', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('generated_islands_grid.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ 岛屿网格已保存！")

## 第五部分：消融实验

In [ ]:
# 对比不同方法的性能
print("运行消融实验...")

# 方法1: 纯随机PCG
print("\n1. 纯随机PCG...")
random_metrics = []
for i in range(20):
    params = {
        'f': np.random.uniform(5, 20),
        'A': np.random.uniform(0.5, 1.5),
        'N_octaves': np.random.randint(3, 6),
        'persistence': np.random.uniform(0.3, 0.7),
        'lacunarity': np.random.uniform(1.5, 2.5),
        'seed': i*100,
        'warp_strength': np.random.uniform(0.2, 0.8),
        'warp_frequency': np.random.uniform(1, 5),
        'falloff_radius': np.random.uniform(20, 40),
        'falloff_exponent': np.random.uniform(1.5, 3)
    }
    hm = generator.generate_heightmap(params)
    metrics = evaluator.evaluate(hm)
    random_metrics.append(metrics)

# 方法2: SAC优化（已有）
sac_metrics = test_metrics_list

# 对比分析
print("\n对比结果:")
print("=" * 70)
print(f"{'Metric':25s} | {'Random PCG':12s} | {'SAC (Ours)':12s} | {'Improvement':12s}")
print("-" * 70)

for key in metric_keys:
    random_vals = [m[key] for m in random_metrics]
    sac_vals = [m[key] for m in sac_metrics]
    
    random_mean = np.mean(random_vals)
    sac_mean = np.mean(sac_vals)
    improvement = ((sac_mean - random_mean) / (random_mean + 1e-8)) * 100
    
    print(f"{key:25s} | {random_mean:12.4f} | {sac_mean:12.4f} | {improvement:+11.2f}%")

In [ ]:
# 可视化对比
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for idx, key in enumerate(metric_keys[:6]):
    random_vals = [m[key] for m in random_metrics]
    sac_vals = [m[key] for m in sac_metrics]
    
    data = [random_vals, sac_vals]
    bp = axes[idx].boxplot(data, labels=['Random PCG', 'SAC (Ours)'], patch_artist=True)
    
    colors = ['lightblue', 'lightcoral']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    
    axes[idx].set_ylabel(key)
    axes[idx].set_title(f'Comparison: {key}')
    axes[idx].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('ablation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ 消融实验完成！")

## 第六部分：最终结果展示

In [ ]:
# 选择最佳岛屿进行详细展示
best_idx = np.argmax([m['connectivity'] * m['navigable_ratio'] for m in test_metrics_list])
best_island = test_islands[best_idx]
best_metrics = test_metrics_list[best_idx]

print(f"最佳岛屿索引: {best_idx + 1}")
print(f"质量评分: {best_metrics['connectivity'] * best_metrics['navigable_ratio']:.4f}")

# 详细可视化
fig = plt.figure(figsize=(18, 6))

# 1. 高度图
ax1 = plt.subplot(141)
im1 = ax1.imshow(best_island, cmap='terrain')
ax1.set_title('Heightmap')
plt.colorbar(im1, ax=ax1)

# 2. 3D地形
ax2 = fig.add_subplot(142, projection='3d')
step = 2
x = np.arange(0, best_island.shape[0], step)
y = np.arange(0, best_island.shape[1], step)
X, Y = np.meshgrid(x, y)
Z = best_island[::step, ::step]
surf = ax2.plot_surface(X, Y, Z, cmap='terrain', edgecolor='none', alpha=0.9)
ax2.set_title('3D Terrain')
ax2.view_init(elev=30, azim=45)

# 3. 二值化（陆地/水域）
binary = (best_island > 0.3).astype(int)
ax3 = plt.subplot(143)
ax3.imshow(binary, cmap='binary')
ax3.set_title('Land/Water Binary')

# 4. 质量指标雷达图
ax4 = fig.add_subplot(144, projection='polar')
categories = list(best_metrics.keys())
values = list(best_metrics.values())

values += values[:1]
angles = [n / float(len(categories)) * 2 * np.pi for n in range(len(categories))]
angles += angles[:1]

ax4.plot(angles, values, 'o-', linewidth=2, color='red')
ax4.fill(angles, values, alpha=0.25, color='red')
ax4.set_xticks(angles[:-1])
ax4.set_xticklabels(categories, fontsize=9)
ax4.set_title('Quality Metrics', pad=20)

plt.tight_layout()
plt.savefig('final_best_island.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 最佳岛屿详细指标:")
print("=" * 50)
for key, value in best_metrics.items():
    print(f"{key:25s}: {value:.4f}")

## 第七部分：保存模型和结果

In [ ]:
# 保存训练好的模型
os.makedirs('saved_models', exist_ok=True)

# 保存VAE
torch.save(vae.state_dict(), 'saved_models/beta_vae.pth')
print("✅ VAE模型已保存到 saved_models/beta_vae.pth")

# 保存SAC
agent.save('saved_models/sac_agent.pth')
print("✅ SAC智能体已保存到 saved_models/sac_agent.pth")

# 保存训练历史
import json
training_info = {
    'vae_loss_history': vae_loss_history,
    'episode_rewards': episode_rewards,
    'episode_lengths': episode_lengths,
    'test_metrics': test_metrics_list
}

with open('saved_models/training_history.json', 'w') as f:
    json.dump(training_info, f, indent=2)
print("✅ 训练历史已保存到 saved_models/training_history.json")

print("\n🎉 完整实验流程结束！")

## 总结

本Notebook完成了以下工作：

1. ✅ **数据集生成**：生成了500个随机岛屿高度图
2. ✅ **β-VAE训练**：训练了隐空间表征模型，实现了高度图的压缩和重建
3. ✅ **SAC训练**：使用Soft Actor-Critic算法优化PCG参数
4. ✅ **模型评估**：生成了20个测试岛屿并进行了统计分析
5. ✅ **消融实验**：对比了随机PCG和SAC优化的性能差异
6. ✅ **结果展示**：展示了最佳生成岛屿的详细可视化

**关键发现**：
- SAC优化显著提升了岛屿的结构质量
- 连通性和可导航比例得到明显改善
- 生成的岛屿具有多样性和可控性

**下一步改进方向**：
- 增加训练episodes和数据集规模
- 引入VAE隐变量到RL状态空间
- 实现更复杂的奖励函数
- 扩展到更大的地图尺寸（128x128或256x256）